# Load Data

In [1]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

df_train_clean = pd.read_csv('data_output/1_data_cleaned_train.csv')
df_test_clean = pd.read_csv('data_output/1_data_cleaned_test.csv')

print("Successfully loaded Train and Test data!")

target_col = 'Khoảng giá' 

X_train = df_train_clean.drop(columns=[target_col])
y_train = df_train_clean[target_col]

X_test = df_test_clean.drop(columns=[target_col])
y_test = df_test_clean[target_col]

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

Successfully loaded Train and Test data!
X_train shape: (4645, 25)
X_test shape: (1164, 25)


In [2]:
print(X_train.columns)

Index(['Diện tích', 'Số phòng ngủ', 'Số phòng tắm, vệ sinh', 'Số tầng',
       'Đường vào', 'Pháp lý', 'Nội thất', 'Latitude', 'Longitude',
       'Quan_Huyen', 'Phuong_Xa', 'Dien_Tich_Lo', 'hem_xe_hoi',
       'gan_cho_sieu_thi', 'gan_truong_hoc', 'gan_benh_vien',
       'gan_cong_vien_ho_nuoc', 'duong_vao_null_flag', 'Duong',
       'Ty_Le_Giao_Thong', 'Ty_Le_Cong_Cong_CX', 'Ty_Le_Khac', 'Ty_Le_Dat_O',
       'Tổng số phòng', 'khoang_cach_trung_tam'],
      dtype='object')


In [3]:
print(X_train['Pháp lý'].unique())
print(X_train['Nội thất'].unique())

['Sổ hồng' 'Sổ đỏ' 'không có']
['Trống / Nhà thô' 'Cơ bản' 'Đầy đủ' 'Cao cấp']


# Encode Categorical Variables: 'Pháp lí', 'Nội thất'

In [4]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

phap_ly_order = ['không có', 'Red Book Certificate', 'Pink Book Certificate']

noi_that_order = ['Bare Shell / Unfurnished', 'Basic Furnished', 'Fully Furnished', 'Premium Furnished']

orde = OrdinalEncoder(
    categories=[phap_ly_order, noi_that_order], 
    handle_unknown='use_encoded_value', 
    unknown_value=-1
)

ord_cols = ['Pháp lý', 'Nội thất']


X_train[ord_cols] = orde.fit_transform(X_train[ord_cols])

X_test[ord_cols] = orde.transform(X_test[ord_cols])

print("✅ Successfully applied Ordinal Encoding to Legal Status and Interior Furnishing!")
print(f"Encoded Legal Status order: {phap_ly_order} -> [0, 1, 2]")
print(f"Encoded Interior Furnishing order: {noi_that_order} -> [0, 1, 2, 3]")

✅ Successfully applied Ordinal Encoding to Legal Status and Interior Furnishing!
Encoded Legal Status order: ['không có', 'Red Book Certificate', 'Pink Book Certificate'] -> [0, 1, 2]
Encoded Interior Furnishing order: ['Bare Shell / Unfurnished', 'Basic Furnished', 'Fully Furnished', 'Premium Furnished'] -> [0, 1, 2, 3]


# Encode Address Variables

In [5]:

thong_ke_duong = X_train['Duong'].value_counts().reset_index()

thong_ke_duong.columns = ['Street Name', 'Record Count']


print("📍 TOP 15 STREETS WITH MOST RECORDS IN TRAINING SET:")
print(thong_ke_duong.head(15))

print("-" * 50)


duong_hiem = thong_ke_duong[thong_ke_duong['Record Count'] <= 2]
print(f"⚠️ Number of streets appearing only 1-2 times: {len(duong_hiem)} streets")

📍 TOP 15 STREETS WITH MOST RECORDS IN TRAINING SET:
                  Street Name  Record Count
0        Đường Huỳnh Tấn Phát           157
1           Đường Quang Trung            50
2     Đường Xô Viết Nghệ Tĩnh            44
3         Đường Nơ Trang Long            44
4             Đường Lê Văn Sỹ            43
5   Đường Cách Mạng Tháng Tám            40
6          Đường Lê Văn Lương            39
7         Đường Lạc Long Quân            37
8          Đường Phan Văn Trị            36
9       Đường Thích Quảng Đức            35
10       Đường Hoàng Hoa Thám            33
11        Đường Điện Biên Phủ            33
12          Đường Nguyễn Trãi            33
13         Đường Phan Huy Ích            32
14        Đường Lê Quang Định            31
--------------------------------------------------
⚠️ Number of streets appearing only 1-2 times: 520 streets


In [11]:
%pip install category_encoders

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
from category_encoders import TargetEncoder


cols_quan = ['Quan_Huyen']
te_quan = TargetEncoder(cols=cols_quan, smoothing=10.0, min_samples_leaf=50)

# Learn and transform Train/Test set
X_train[cols_quan] = te_quan.fit_transform(X_train[cols_quan], y_train)
X_test[cols_quan] = te_quan.transform(X_test[cols_quan])


cols_phuong_duong = ['Phuong_Xa', 'Duong']
te_phuong_duong = TargetEncoder(cols=cols_phuong_duong, smoothing=10.0, min_samples_leaf=5)

X_train[cols_phuong_duong] = te_phuong_duong.fit_transform(X_train[cols_phuong_duong], y_train)
X_test[cols_phuong_duong] = te_phuong_duong.transform(X_test[cols_phuong_duong])


print("✅ Successfully applied Target Encoding!")
print("  - 'District' column: min_samples_leaf = 50")
print("  - 'Ward/Commune/Town' & 'Street_From_Address' columns: min_samples_leaf = 5")

✅ Successfully applied Target Encoding!
  - 'District' column: min_samples_leaf = 50
  - 'Ward/Commune/Town' & 'Street_From_Address' columns: min_samples_leaf = 5


# Save Data

In [ ]:

X_train_tree = X_train.copy()
X_test_tree = X_test.copy()

df_train_tree = pd.concat([X_train_tree, y_train], axis=1)
df_test_tree = pd.concat([X_test_tree, y_test], axis=1) if 'y_test' in locals() else X_test_tree.copy()

df_train_tree.to_csv('data_output/2_data_preprocessed_tree_train.csv', index=False)
df_test_tree.to_csv('data_output/2_data_preprocessed_tree_test.csv', index=False)

print("💾 FILE 2: Successfully saved preprocessed data for Tree-based models!")
print("   -> data_output/2_data_preprocessed_tree_train.csv")
print("   -> data_output/2_data_preprocessed_tree_test.csv")

💾 FILE 2: Đã lưu dữ liệu cho mô hình dạng Cây thành công!
   -> data_output/2_data_preprocessed_tree_train.csv
   -> data_output/2_data_preprocessed_tree_test.csv


In [ ]:
print(X_train.columns)

Index(['Diện tích', 'Số phòng ngủ', 'Số phòng tắm, vệ sinh', 'Số tầng',
       'Đường vào', 'Pháp lý', 'Nội thất', 'Latitude', 'Longitude',
       'Quan_Huyen', 'Phuong_Xa', 'Dien_Tich_Lo', 'hem_xe_hoi',
       'gan_cho_sieu_thi', 'gan_truong_hoc', 'gan_benh_vien',
       'gan_cong_vien_ho_nuoc', 'duong_vao_null_flag', 'Duong',
       'Ty_Le_Giao_Thong', 'Ty_Le_Cong_Cong_CX', 'Ty_Le_Khac', 'Ty_Le_Dat_O',
       'Tổng số phòng', 'khoang_cach_trung_tam'],
      dtype='object')
